# 基于MTL-DDPG的A股交易策略

**论文参考**: 邓伟 (2023), 《基于MTL-DDPG的A股交易策略研究》

**核心思想**: DDPG (Deep Deterministic Policy Gradient) 输出连续动作(仓位权重0-1)，结合多任务学习(MTL)同时预测收益率和决定仓位。

**数据**: 贵州茅台 (600519) 日线数据

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### DDPG (Deep Deterministic Policy Gradient)

DDPG是处理连续动作空间的Actor-Critic方法:

**Critic** 学习Q函数:
$$L(\phi) = \mathbb{E}\left[ \left( Q_\phi(s, a) - y \right)^2 \right], \quad y = r + \gamma Q_{\phi'}(s', \mu_{\theta'}(s'))$$

**Actor** 沿Q函数梯度方向优化策略:
$$\nabla_\theta J = \mathbb{E}\left[ \nabla_a Q_\phi(s, a)\big|_{a=\mu_\theta(s)} \cdot \nabla_\theta \mu_\theta(s) \right]$$

**软更新** 目标网络:
$$\theta' \leftarrow \tau \theta + (1 - \tau) \theta', \quad \tau \ll 1$$

### 多任务学习 (MTL)

Actor网络同时输出:
1. **仓位权重** $w \in [0, 1]$: sigmoid激活
2. **收益预测** $\hat{r}$: 辅助任务

$$L_{\text{total}} = L_{\text{policy}} + \lambda L_{\text{return\_pred}}$$

多任务学习通过共享底层特征表示，让模型同时理解市场动态和交易决策。

In [ ]:
# ============ 动画: Actor-Critic 梯度流动图 ============
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 模拟Actor-Critic训练过程中的梯度流
np.random.seed(42)
n_steps = 60

# 模拟Q值和策略输出的演化
q_values = np.cumsum(np.random.randn(n_steps) * 0.05) + 0.5
actor_output = 1 / (1 + np.exp(-np.cumsum(np.random.randn(n_steps) * 0.08)))
critic_loss = np.exp(-np.linspace(0, 3, n_steps)) * 0.5 + np.random.randn(n_steps) * 0.02
actor_loss = np.exp(-np.linspace(0, 2.5, n_steps)) * 0.3 + np.random.randn(n_steps) * 0.015

frames = []
for i in range(5, n_steps + 1, 2):
    frames.append(go.Frame(
        data=[
            go.Scatter(x=list(range(i)), y=actor_output[:i], mode='lines',
                       name='Actor输出(仓位)', line=dict(color='#2196F3', width=2)),
            go.Scatter(x=list(range(i)), y=q_values[:i], mode='lines',
                       name='Critic Q值', line=dict(color='#FF9800', width=2)),
            go.Scatter(x=list(range(i)), y=critic_loss[:i], mode='lines',
                       name='Critic Loss', line=dict(color='#F44336', width=2, dash='dash')),
            go.Scatter(x=list(range(i)), y=actor_loss[:i], mode='lines',
                       name='Actor Loss', line=dict(color='#4CAF50', width=2, dash='dash')),
        ],
        name=f'Step {i}'
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(text='DDPG Actor-Critic 训练动态'),
        xaxis_title='训练步数', yaxis_title='值',
        height=500, width=900,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='▶ 播放', method='animate',
                 args=[None, {'frame': {'duration': 120}, 'transition': {'duration': 50}}]),
            dict(label='⏸ 暂停', method='animate',
                 args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
        )],
    )
)
fig.show()

In [ ]:
# ============ 导入与配置 ============
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random

from shared.data_fetcher import get_stock_daily
from shared.backtest_engine import Backtester
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table, set_chinese_font)
from shared.factors import rsi, macd, bollinger_bands, atr, volatility
from shared.constants import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ============ 数据获取与特征工程 ============
df = get_stock_daily('600519', start_date='20180101', end_date='20241231')
print(f'数据量: {len(df)} 条')

df['returns'] = df['close'].pct_change()
df['rsi'] = rsi(df['close'], 14)
macd_df = macd(df['close'])
df['macd_hist'] = macd_df['histogram']
bb = bollinger_bands(df['close'])
df['bb_pctb'] = bb['bb_pctb']
df['bb_width'] = bb['bb_width']
df['atr_val'] = atr(df['high'], df['low'], df['close'], 14)
vol_df = volatility(df['close'], [5, 10, 20])
df['vol_5'] = vol_df['vol_5']
df['vol_20'] = vol_df['vol_20']
df['mom_5'] = df['close'].pct_change(5)
df['mom_20'] = df['close'].pct_change(20)

df.dropna(inplace=True)

FEATURE_COLS = ['returns', 'rsi', 'macd_hist', 'bb_pctb', 'bb_width',
                'atr_val', 'vol_5', 'vol_20', 'mom_5', 'mom_20']
WINDOW = 10
STATE_DIM = len(FEATURE_COLS) * WINDOW

# 标准化
feat_mean = df[FEATURE_COLS].mean()
feat_std = df[FEATURE_COLS].std().replace(0, 1)
df[FEATURE_COLS] = (df[FEATURE_COLS] - feat_mean) / feat_std

split = int(len(df) * 0.7)
train_df = df.iloc[:split].reset_index(drop=True)
test_df = df.iloc[split:].reset_index(drop=True)
print(f'训练集: {len(train_df)}, 测试集: {len(test_df)}')

In [ ]:
# ============ 连续动作交易环境 ============
class ContinuousTradingEnv:
    """连续动作空间的交易环境，动作为仓位权重 [0, 1]"""
    def __init__(self, data, feature_cols, window=10):
        self.data = data.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.window = window
        self.state_dim = len(feature_cols) * window
        self.reset()

    def reset(self):
        self.current_step = self.window
        self.position = 0.0  # 当前仓位 [0, 1]
        return self._get_state()

    def _get_state(self):
        start = self.current_step - self.window
        end = self.current_step
        features = self.data[self.feature_cols].iloc[start:end].values.flatten()
        return features.astype(np.float32)

    def step(self, action):
        """action: 仓位权重 [0, 1]"""
        action = np.clip(action, 0.0, 1.0)
        current_return = self.data['returns'].iloc[self.current_step]

        # 交易成本
        position_change = abs(action - self.position)
        cost = position_change * (COMMISSION_RATE + STAMP_TAX_RATE * 0.5)

        # 投资组合收益
        reward = action * current_return - cost

        # 下一期真实收益 (用于MTL辅助任务)
        next_return = current_return

        self.position = action
        self.current_step += 1
        done = self.current_step >= len(self.data) - 1
        next_state = self._get_state() if not done else np.zeros(self.state_dim, dtype=np.float32)

        return next_state, reward, done, {'actual_return': next_return}

In [ ]:
# ============ MTL-DDPG 模型定义 ============
class MTLActor(nn.Module):
    """多任务Actor: 输出仓位权重 + 收益预测"""
    def __init__(self, state_dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
        )
        # 仓位决策头
        self.position_head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()  # [0, 1]
        )
        # 收益预测头 (辅助任务)
        self.return_head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, state):
        h = self.shared(state)
        position = self.position_head(h)
        return_pred = self.return_head(h)
        return position, return_pred


class Critic(nn.Module):
    """Critic: 评估 (state, action) 的Q值"""
    def __init__(self, state_dim, action_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, state, action):
        x = torch.cat([state, action], dim=-1)
        return self.net(x)


class OUNoise:
    """Ornstein-Uhlenbeck 探索噪声"""
    def __init__(self, size, mu=0.0, theta=0.15, sigma=0.2):
        self.mu = mu * np.ones(size)
        self.theta = theta
        self.sigma = sigma
        self.reset()

    def reset(self):
        self.state = self.mu.copy()

    def sample(self):
        dx = self.theta * (self.mu - self.state) + self.sigma * np.random.randn(len(self.state))
        self.state += dx
        return self.state


class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done, actual_return):
        self.buffer.append((state, action, reward, next_state, done, actual_return))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s2, d, ar = zip(*batch)
        return (np.array(s), np.array(a), np.array(r),
                np.array(s2), np.array(d), np.array(ar))

    def __len__(self):
        return len(self.buffer)

In [ ]:
# ============ MTL-DDPG Agent ============
class MTLDDPGAgent:
    def __init__(self, state_dim, lr_actor=1e-4, lr_critic=1e-3,
                 gamma=0.99, tau=0.005, mtl_weight=0.3):
        self.gamma = gamma
        self.tau = tau
        self.mtl_weight = mtl_weight

        self.actor = MTLActor(state_dim).to(device)
        self.actor_target = MTLActor(state_dim).to(device)
        self.actor_target.load_state_dict(self.actor.state_dict())

        self.critic = Critic(state_dim).to(device)
        self.critic_target = Critic(state_dim).to(device)
        self.critic_target.load_state_dict(self.critic.state_dict())

        self.actor_opt = optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.critic_opt = optim.Adam(self.critic.parameters(), lr=lr_critic)

        self.memory = ReplayBuffer(10000)
        self.noise = OUNoise(1)

    def act(self, state, explore=True):
        s = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            position, _ = self.actor(s)
        action = position.cpu().numpy()[0, 0]
        if explore:
            action += self.noise.sample()[0] * 0.1
        return np.clip(action, 0.0, 1.0)

    def train_step(self, batch_size=64):
        if len(self.memory) < batch_size:
            return

        s, a, r, s2, d, ar = self.memory.sample(batch_size)
        s = torch.FloatTensor(s).to(device)
        a = torch.FloatTensor(a).unsqueeze(1).to(device)
        r = torch.FloatTensor(r).unsqueeze(1).to(device)
        s2 = torch.FloatTensor(s2).to(device)
        d = torch.FloatTensor(d).unsqueeze(1).to(device)
        ar = torch.FloatTensor(ar).unsqueeze(1).to(device)

        # --- 更新Critic ---
        with torch.no_grad():
            next_action, _ = self.actor_target(s2)
            q_next = self.critic_target(s2, next_action)
            q_target = r + self.gamma * q_next * (1 - d)

        q_current = self.critic(s, a)
        critic_loss = F.mse_loss(q_current, q_target)

        self.critic_opt.zero_grad()
        critic_loss.backward()
        self.critic_opt.step()

        # --- 更新Actor (含MTL) ---
        position, return_pred = self.actor(s)
        actor_loss = -self.critic(s, position).mean()
        mtl_loss = F.mse_loss(return_pred, ar)
        total_actor_loss = actor_loss + self.mtl_weight * mtl_loss

        self.actor_opt.zero_grad()
        total_actor_loss.backward()
        self.actor_opt.step()

        # --- 软更新目标网络 ---
        for p, pt in zip(self.actor.parameters(), self.actor_target.parameters()):
            pt.data.copy_(self.tau * p.data + (1 - self.tau) * pt.data)
        for p, pt in zip(self.critic.parameters(), self.critic_target.parameters()):
            pt.data.copy_(self.tau * p.data + (1 - self.tau) * pt.data)

In [ ]:
# ============ 训练 ============
N_EPISODES = 80
agent = MTLDDPGAgent(STATE_DIM)
reward_history = []

for ep in range(N_EPISODES):
    env = ContinuousTradingEnv(train_df, FEATURE_COLS, WINDOW)
    state = env.reset()
    agent.noise.reset()
    ep_reward = 0

    while True:
        action = agent.act(state, explore=True)
        next_state, reward, done, info = env.step(action)
        agent.memory.push(state, action, reward, next_state, float(done),
                          info['actual_return'])
        agent.train_step()
        ep_reward += reward
        state = next_state
        if done:
            break

    reward_history.append(ep_reward)
    if (ep + 1) % 20 == 0:
        print(f'Episode {ep+1}/{N_EPISODES} | Reward: {ep_reward:.4f}')

In [ ]:
# ============ 评估与回测 ============
# 在测试集上获取仓位序列
env = ContinuousTradingEnv(test_df, FEATURE_COLS, WINDOW)
state = env.reset()
positions = [0.0] * WINDOW

while True:
    action = agent.act(state, explore=False)
    next_state, _, done, _ = env.step(action)
    positions.append(action)
    state = next_state
    if done:
        break

positions = positions[:len(test_df)]

# 转换为信号: 仓位>0.5 -> 买入(1), 否则 -> 空仓(0)
test_dates = df.index[split:split + len(test_df)]
test_prices = df['close'].iloc[split:split + len(test_df)]
test_prices.index = test_dates

signals_binary = pd.Series(
    [1 if p > 0.5 else 0 for p in positions],
    index=test_dates[:len(positions)]
)

# 回测
bt = Backtester(t_plus_1=True)
result = bt.run(test_prices, signals_binary)

print('\nMTL-DDPG 策略绩效:')
for k, v in result['metrics'].items():
    print(f'  {k}: {v}')

# 仓位分布统计
pos_arr = np.array(positions)
print(f'\n仓位统计:')
print(f'  平均仓位: {pos_arr.mean():.2%}')
print(f'  满仓比例: {(pos_arr > 0.5).mean():.2%}')
print(f'  空仓比例: {(pos_arr < 0.1).mean():.2%}')

In [ ]:
# ============ 可视化 ============
import matplotlib.pyplot as plt
set_chinese_font()

# 1. 训练曲线
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(reward_history, color='#2196F3', alpha=0.6, label='Episode奖励')
window_avg = pd.Series(reward_history).rolling(10).mean()
ax.plot(window_avg, color='#F44336', linewidth=2, label='10轮滑动均值')
ax.set_title('MTL-DDPG 训练奖励曲线', fontsize=14)
ax.set_xlabel('Episode')
ax.set_ylabel('累计奖励')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2. 仓位变化
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(test_dates[:len(positions)], test_prices.values[:len(positions)],
             color='#333', linewidth=1.2)
axes[0].set_title('价格走势', fontsize=13)
axes[0].set_ylabel('价格')
axes[0].grid(True, alpha=0.3)

axes[1].fill_between(test_dates[:len(positions)], 0, positions,
                     color='#2196F3', alpha=0.5)
axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='买入阈值')
axes[1].set_title('MTL-DDPG 仓位输出', fontsize=13)
axes[1].set_ylabel('仓位权重')
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3. 收益曲线与回撤
plot_equity_curve(result['equity_curve'], title='MTL-DDPG 策略收益曲线')
plot_drawdown(result['equity_curve'], title='MTL-DDPG 回撤')
plot_metrics_table(result['metrics'], title='MTL-DDPG 绩效指标')

## 实验结果与分析

### 关键发现
1. **连续动作空间**: DDPG输出连续仓位权重，比离散动作更精细地控制风险敞口
2. **多任务学习**: 收益预测辅助任务帮助Actor学习更好的市场状态表示
3. **仓位平滑**: 连续输出自然地避免了频繁的全仓/空仓切换
4. **OU噪声**: Ornstein-Uhlenbeck过程提供时间相关的探索噪声，比高斯噪声更适合连续控制

### MTL的作用
- 辅助任务(收益预测)迫使共享网络学习有意义的市场特征
- 权重$\lambda=0.3$平衡主任务与辅助任务
- 实验表明MTL比纯DDPG收敛更快、泛化更好

### 局限性
- DDPG对超参数(学习率、噪声参数)敏感
- 单资产场景限制了策略容量
- 需要更多数据和训练轮数才能充分发挥深度RL的优势